In [57]:
import torch 
from d2l import torch as d2l

In [58]:
def corr2d_multi_in(X, K):
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

In [59]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
X, K, K.shape

(tensor([[[0., 1., 2.],
          [3., 4., 5.],
          [6., 7., 8.]],
 
         [[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]),
 tensor([[[0., 1.],
          [2., 3.]],
 
         [[1., 2.],
          [3., 4.]]]),
 torch.Size([2, 2, 2]))

In [60]:
corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

多输出通道

In [61]:
def corr2d_multi_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [62]:
K = torch.stack((K,K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [63]:
corr2d_multi_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

1 * 1卷积层

In [64]:
def corr2d_multi_in_out_11(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))


In [65]:
X = torch.normal(0, 1, (3, 2, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
X, K

(tensor([[[ 0.5590, -1.0511, -0.2681],
          [ 0.2273, -0.0911,  0.0822]],
 
         [[ 1.5124, -1.4923,  0.8579],
          [-0.3626,  1.6889, -0.4437]],
 
         [[ 0.3740,  0.7517, -0.7929],
          [ 0.9630, -0.1994,  0.6705]]]),
 tensor([[[[ 0.7419]],
 
          [[-1.7970]],
 
          [[-1.5968]]],
 
 
         [[[ 0.4683]],
 
          [[-1.2681]],
 
          [[-1.1212]]]]))

In [66]:
Y1 = corr2d_multi_in_out_11(X, K)
Y2 = corr2d_multi_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6